In [1]:
#GNU General Public License v3.0
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy.io.wavfile import read #import the required function from the module
from scipy.fftpack import fft, fftfreq

import math

import os
from pathlib import Path

import librosa

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

#Dataframe X dimensions (event, sensors (1), readings(1000)
#Dataframe Y dimensions (Alarm/Not)


In [2]:
#convert time series data into freqency domain with scipy fft

def fft_check(filename):
    global event_count, pa_x, pa_y, first_run, freqs_al, mag_al, freqs_no, mag_no

    sect_data = [] #pd.DataFrame()
    sect_dur = 0.124 # resulted in 94 M_RS
    #sect_dur = 2.48 #resulted in 88.
    #went down with just freqs and no mag
    
    path = 'C:/Users/Temp Sound Project/wav_files/' + filename
    
    samplerate, data = read(path)  #read(path) #'.wav')

    if len(data.shape) > 1:
        data = data[:,0]
        #print("len(data.shape): ", len(data.shape))

    data = data[799:]
    #print(data)


    tot_duration = len(data)/samplerate
    #print(data)
    
    sects = math.floor(tot_duration/sect_dur)
    
    #print(filename, " ", tot_duration, " ", sects)

    for i in range(0, sects):
        
        #event number
        event_num = event_count
        sect_data = []
        temp_data = []
        
        #sensors
        sensors = 1

        temp_data = data[ round(i*sect_dur*samplerate):round((i+1)*sect_dur*samplerate) - 1]
        
        fft_out = fft(temp_data)
        freqs = fftfreq(len(temp_data), 1/samplerate)
        #print("Frequencies:", freqs[:10])
        #print("FFT magnitudes:", np.abs(fft_out)[:10])
    
        sect_data = np.vstack((freqs[:4000], np.abs(fft_out)[:4000]))  #88
        
        sect_duration = len(sect_data)/samplerate
        time = np.arange(0, sect_duration,1/samplerate) #time vector
    
        if len(time) > len(sect_data):
            time = time[:-1]

        np.array(sect_data)
        
        if first_run == True:
            pa_x = sect_data
            np.array(pa_x)
            first_run = False
        else:
            #print("vstack")
            new_x = np.vstack((pa_x, sect_data))
            pa_x = new_x
            np.array(pa_x)

              
        #y_value
        y_val = "No"
        if "M_RS" in filename or "sawzall" in filename:
            y_val = "Alarm"
            freqs_al = np.append(freqs_al, freqs[:4000]) #:20])
            mag_al =  np.append(mag_al, np.abs(fft_out)[:4000]) #:20])
        else:
            freqs_no = np.append(freqs_no, freqs[:4000]) #:20])
            mag_no =  np.append(mag_no, np.abs(fft_out)[:4000]) #:20])
            
        pa_y.append(y_val) #pa_y.loc[len(df_y)] = y_val

        #print("----", filename)
        # plt.scatter(freqs[:4000], np.abs(fft_out)[:4000])
        # plt.show()

In [3]:
pa_x =[]
pa_y =[]

freqs_al = []
freqs_no = [] 
mag_al = [] 
mag_no = []

path = 'C:/Users/Temp Sound Project/wav_files'

p = Path(path)
event_count = 0
first_run = True

for i in p.glob('**/*'):
     print(i.name)
     event_count += 1
    
     fft_check(i.name)
     file_to_df_flag = False
    
     first_run = False
     #break

pa_x = np.array(pa_x)
pa_y = np.array(pa_y)

print("end")
print(pa_x)
print(pa_y)
#df_3d = xarray_3d.to_dataframe()

print("freqs_al", freqs_al)
print("mag_al", mag_al)
print("freqs_no", freqs_no)
print("mag_no", mag_no)


Alarm file-2022.7.19-20.29.7.wav
Alarm file-2022.7.20-19.23.12.wav
Alarm file-2022.7.21-17.56.2.wav
Alarm file-2022.7.21-18.9.32.wav
Alarm file-2022.7.23-10.7.4 passing.wav
Alarm file-2022.7.23-12.31.33.wav
Alarm file-2022.7.23-18.42.27 car passing.wav
Alarm file-2022.7.24-13.17.48.wav
Alarm file-2022.7.28-18.23.42.wav
Alarm file-2022.7.28-6.52.26_car_wet_pavement.wav
Alarm file-2022.7.29-10.31.2.wav
Alarm file-2022.7.29-21.45.9.wav
Alarm file-2022.7.30-14.22.49.wav
Alarm file-2022.7.30-19.38.48.wav
Alarm_file-2022.7.31-14.11.17.wav
Alarm_file-2022.7.31-17.2.45.wav
Alarm_file-2022.7.31-17.54.38.wav
Alarm_file-2022.8.1-14.23.17_sportscar_starting.wav
Alarm_file-2022.8.1-15.27.43.wav
Alarm_file-2022.8.16-6.33.28.wav
Alarm_file-2022.8.16-6.44.50.wav
Alarm_file-2022.8.2-15.31.42.wav
Alarm_file-2022.8.2-7.22.24.wav
Alarm_file-2022.8.4-11.40.11_ups_truck.wav
Alarm_file-2022.8.4-11.42.51_objective_signal.wav
Alarm_file-2022.8.4-11.56.7.wav
Alarm_file-2022.8.4-13.52.19.wav
Alarm_file-2022.8.5-

In [4]:
if file_to_df_flag != True:
    print("pa_x: ", pa_x.shape)
    #print(pa_x)

    #remove odd event if applicable
    if pa_x.shape[0] % 2 != 0:
        pa_x = pa_x[:-1,:]
        pa_y = pa_y[:-1]
        print("   pa_x: ", pa_x.shape)
        print("   pa_y: ", pa_y.shape)

    #reshape 2D array into a 3D arra
    pa_x = pa_x.reshape((int(pa_x.shape[0]/2), 2, pa_x.shape[1]))
    print("pa_x reshape: ", pa_x.shape)
    print("pa_y: ", pa_y.shape)

    #print(pa_x)

pa_x:  (7510, 4000)
pa_x reshape:  (3755, 2, 4000)
pa_y:  (3755,)


In [5]:
#knn classifier
from sktime.classification.distance_based import KNeighborsTimeSeriesClassifier

knn_clf = KNeighborsTimeSeriesClassifier(
    n_neighbors=3,
    distance='euclidean'
)

X_train, X_test, y_train, y_test = train_test_split(
    pa_x, pa_y, test_size=0.33, random_state=42)


knn_clf.fit(X_train, y_train)

KNeighborsTimeSeriesClassifier(distance='euclidean', n_neighbors=3)

In [6]:
y_pred = knn_clf.predict(X_test)

from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.9854838709677419

In [8]:
%%time

#Shapelet classifier
from sktime.classification.shapelet_based import ShapeletTransformClassifier

from sklearn.metrics import classification_report, precision_recall_fscore_support 

clf = ShapeletTransformClassifier(
    n_shapelet_samples=100,
    max_shapelets=100,
    time_limit_in_minutes=5,
    contract_max_n_shapelet_samples=100,
    n_jobs=-1,
    random_state=42
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

clf_report = classification_report(y_test, y_pred)

print(clf_report)

C:\Users\ChrisChew\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\ChrisChew\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


              precision    recall  f1-score   support

       Alarm       0.95      0.96      0.95       521
          No       0.97      0.96      0.97       719

    accuracy                           0.96      1240
   macro avg       0.96      0.96      0.96      1240
weighted avg       0.96      0.96      0.96      1240

CPU times: total: 3min 49s
Wall time: 13min 20s


In [15]:
#SVC Classifier
print(X_train.shape)
X_tr_reshaped_array= X_train.reshape(X_train.shape[0], (X_train.shape[1]*X_train.shape[2]))
print(X_tr_reshaped_array.shape)
X_te_reshaped_array= X_test.reshape(X_test.shape[0], (X_test.shape[1]*X_test.shape[2]))

(2515, 2, 4000)
(2515, 8000)


In [16]:
from sklearn import svm

model = svm.SVC()  # Create an SVM model
model.fit(X_tr_reshaped_array, y_train)

SVC()

In [17]:
predictions = model.predict(X_te_reshaped_array)  # Predict on the test data

In [18]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, predictions)
print(f'Accuracy: {accuracy}')

Accuracy: 0.9387096774193548
